In [1]:
import psycopg2
import json
import csv
import pandas as pd
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values

In [25]:
def connect_to_postgres(dbname=None):
    """Connect to PostgreSQL. If dbname is None, connect to the default database."""
    return psycopg2.connect(
        dbname=dbname or "postgres",
        user="postgres",
        password="123456",
        host="localhost",
        port="5432"
    )

def create_database_if_not_exists(dbname):
    """Create the database if it does not exist."""
    conn = connect_to_postgres()
    conn.autocommit = True
    try:
        with conn.cursor() as cur:
            cur.execute(
                sql.SQL("CREATE DATABASE {};").format(sql.Identifier(dbname))
            )
            print(f"Database '{dbname}' created successfully.")
    except psycopg2.errors.DuplicateDatabase:
        print(f"Database '{dbname}' already exists.")
    finally:
        conn.close()

def create_tables_if_not_exists(conn):
    """Create tables in the database if they do not exist."""
    table_creation_queries = {
    "adidas_sales": """
        CREATE TABLE IF NOT EXISTS adidas_sales (
        retailer TEXT NOT NULL,
        retailer_id INT NOT NULL,
        invoice_date TIMESTAMP NOT NULL,
        region TEXT NOT NULL,
        state TEXT NOT NULL,
        city TEXT NOT NULL,
        product TEXT NOT NULL,
        price_per_unit NUMERIC(10, 2) NOT NULL,
        units_sold INT NOT NULL,
        total_sales NUMERIC(15, 2) NOT NULL,
        operating_profit NUMERIC(15, 2) NOT NULL,
        operating_margin NUMERIC(5, 2) NOT NULL,
        sales_method TEXT NOT NULL
    );

    """
    }
    
    with conn.cursor() as cur:
        for table, query in table_creation_queries.items():
            cur.execute(query)
        conn.commit()
    print("Tables created successfully (if they did not already exist).")

def insert_data_to_postgres(conn, table_name, data):
    """Insert data into a PostgreSQL table."""
    with conn.cursor() as cur:
        columns = data.columns.tolist()
        values = [tuple(x) for x in data.to_numpy()]
        insert_query = f"""
        INSERT INTO {table_name} ({', '.join(columns)}) 
        VALUES %s
        """
        execute_values(cur, insert_query, values)
        conn.commit()

In [26]:
# Define the database name
db_name = "adidas_sales"

# Create the database if it doesn't exist
create_database_if_not_exists(db_name)

# Connect to the newly created database
conn = connect_to_postgres(dbname=db_name)

# Ensure tables exist
create_tables_if_not_exists(conn)


Database 'adidas_sales' created successfully.
Tables created successfully (if they did not already exist).


In [27]:
adidas_df = pd.read_csv('CLEANED Adidas US Sales Dataset.csv')
adidas_df

,Retailer,Retailer ID,Invoice Date,Region,State,City,Product,Price per Unit,Units Sold,Total Sales,Operating Profit,Operating Margin,Sales Method
0,Foot Locker,1185732,2020-01-01 00:00:00,Northeast,New York,New York,Men's Street Footwear,50.0,1200,600000.0,300000.00,0.50,In-store
1,Foot Locker,1185732,2020-01-02 00:00:00,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,500000.0,150000.00,0.30,In-store
2,Foot Locker,1185732,2020-01-03 00:00:00,Northeast,New York,New York,Women's Street Footwear,40.0,1000,400000.0,140000.00,0.35,In-store
3,Foot Locker,1185732,2020-01-04 00:00:00,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,382500.0,133875.00,0.35,In-store
4,Foot Locker,1185732,2020-01-05 00:00:00,Northeast,New York,New York,Men's Apparel,60.0,900,540000.0,162000.00,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9643,Foot Locker,1185732,2021-01-24 00:00:00,Northeast,New Hampshire,Manchester,Men's Apparel,50.0,64,3200.0,896.00,0.28,Outlet
9644,Foot Locker,1185732,2021-01-24 00:00:00,Northeast,New Hampshire,Manchester,Women's Apparel,41.0,105,4305.0,1377.60,0.32,Outlet
9645,Foot Locker,1185732,2021-02-22 00:00:00,Northeast,New Hampshire,Manchester,Men's Street Footwear,41.0,184,7544.0,2791.28,0.37,Outlet
9646,Foot Locker,1185732,2021-02-22 00:00:00,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,42.0,70,2940.0,1234.80,0.42,Outlet


In [28]:
adidas_df.columns = [
    "retailer", "retailer_id", "invoice_date", "region", "state", "city", 
    "product", "price_per_unit", "units_sold", "total_sales", 
    "operating_profit", "operating_margin", "sales_method"
]


In [29]:
try:
    # Transfer data to PostgreSQL
    insert_data_to_postgres(conn, "adidas_sales", adidas_df)
    print("Data successfully transferred to PostgreSQL database.")
finally:
    conn.close()

Data successfully transferred to PostgreSQL database.
